<img src="https://hilpisch.com/tpq_logo_bic.png" width="20%" align="right">

# Python & AI for Algorithmic Trading
## Chapter 15 · IG Broker APIs and EUR/USD Diagnostics

&copy; Dr. Yves J. Hilpisch<br>
AI-Powered by GPT 5.1<br>
The Python Quants GmbH | https://tpq.io<br>
https://hilpisch.com | https://linktr.ee/dyjh


### Notebook Goals

This notebook parallels the IG-market diagnostics using a simple offline example.

- Construct a price series that mimics IG candle data.
- Derive daily returns and risk metrics.
- Plot price and return behaviour for comparison with Oanda results.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.style.use("seaborn-v0_8")  # set notebook-wide plotting style
try:
    from matplotlib_inline.backend_inline import set_matplotlib_formats
    set_matplotlib_formats("svg")
except Exception:
    pass
plt.rcParams.update({"figure.dpi": 300})


### 1. Synthetic IG-Style Prices

We generate an EUR/USD-like series with modest volatility and use it as a stand-in for IG candles.

In [ ]:
rng = np.random.default_rng(seed=15)
dates = pd.date_range("2024-01-01", periods=260, freq="B")
prices = 1.08 + 0.015 * np.cumsum(rng.normal(0.0, 0.01, size=dates.shape[0]))
series = pd.Series(prices, index=dates, name="EUR/USD_IG")
series.head()


### 2. Returns and Simple Risk Metrics

We compute daily simple returns, annualised volatility, and a crude maximum drawdown to match the Chapter 15 narrative.

In [ ]:
rets = series.pct_change(fill_method=None).dropna()
wealth = (1.0 + rets).cumprod()

ann_vol = float(rets.std(ddof=1) * np.sqrt(252.0))
drawdown = wealth / wealth.cummax() - 1.0
max_dd = float(drawdown.min())

print(f"Annualised volatility: {ann_vol: .4f}")
print(f"Maximum drawdown    : {max_dd: .4f}")


### 3. Price and Return Figures

The figure below mirrors the daily diagnostic plots used in the IG chapter.

In [ ]:
fig, (ax_price, ax_ret) = plt.subplots(2, 1, figsize=(7.0, 4.6))

normed = series / series.iloc[0]
ax_price.plot(normed.index, normed.values, color="tab:blue")
ax_price.set_ylabel("price (normalised)")
ax_price.set_title("EUR/USD: IG-style synthetic daily diagnostics")
ax_price.grid(True, alpha=0.3)

ax_ret.hist(rets.values, bins=60, color="tab:green", alpha=0.8)
ax_ret.set_xlabel("daily simple return")
ax_ret.set_ylabel("frequency")
ax_ret.grid(True, alpha=0.3)

fig.tight_layout()
plt.show()


### Takeaways

- Broker-specific chapters share a common core: candle data, returns, and straightforward diagnostics.
- Using synthetic data keeps notebooks runnable without live credentials while preserving the structure of the real workflow.
- When IG access is configured, replace `series` with data coming from the wrapper and reuse all analysis cells unchanged.


<img src="https://hilpisch.com/tpq_logo_bic.png" width="20%" align="right">